In [78]:
import pandas as pd
from tqdm import tqdm
import os
import time
import requests
import gzip
import ast
import numpy as np
import re

In [2]:
data_path='/projects/ccnr/sebek.m/CPE_data/'

### Download and Prepare PubChem Reference Standard Data

In [6]:
pubchem_dir = os.path.join(data_path, 'pubchem')
os.makedirs(pubchem_dir, exist_ok=True)

#### Get PubChem SID - CID mappings

In [18]:
# Download SID-Map file
url = "https://ftp.ncbi.nlm.nih.gov/pubchem/Substance/Extras/SID-Map.gz"
response = requests.head(url)
total_size = int(response.headers.get('content-length', 0))

# Download with progress bar
response = requests.get(url, stream=True)
progress_bar = tqdm(total=total_size, unit='iB', unit_scale=True, desc="Downloading")

# Save data
temp_file = os.path.join(pubchem_dir, 'SID-Map.gz')
with open(temp_file, 'wb') as f:
    for chunk in response.iter_content(chunk_size=8192):
        if chunk:
            progress_bar.update(len(chunk))
            f.write(chunk)
progress_bar.close()

print(f"Saved to {temp_file}")

Downloading: 100%|██████████| 3.51G/3.51G [00:54<00:00, 64.6MiB/s]

Saved to /projects/ccnr/sebek.m/CPE_data/pubchem/SID-Map.gz


In [23]:
temp_file = os.path.join(pubchem_dir, 'SID-Map.gz')

data_4col = []  # Only consider entries with SID, Source, External_ID, and CID
with gzip.open(temp_file, 'rt') as f:
    for line_num, line in enumerate(f, 1):
        parts = line.strip().split('\t')
        
        if len(parts) == 3:
            des='No CID for substance'
        elif len(parts) == 4:
            data_4col.append(parts)
        
        # Progress update
        if line_num % 5000000 == 0:
            print(f"Processed {line_num:,} lines")

# Create DataFrame
pc_sid = pd.DataFrame(data_4col, columns=['SID', 'Source', 'External_ID', 'CID'])
pc_sid['SID'] = pc_sid['SID'].astype(int)
pc_sid['CID'] = pc_sid['CID'].astype(int)

print(f"\nDataFrame with CID mappings: {len(pc_sid):,} rows")

Loading SID-Map with variable column handling...
Processed 5,000,000 lines - 3col: 28,661, 4col: 4,971,339
Processed 10,000,000 lines - 3col: 53,388, 4col: 9,946,612
Processed 15,000,000 lines - 3col: 64,049, 4col: 14,935,951
Processed 20,000,000 lines - 3col: 100,376, 4col: 19,899,624
Processed 25,000,000 lines - 3col: 146,941, 4col: 24,853,059
Processed 30,000,000 lines - 3col: 154,497, 4col: 29,845,503
Processed 35,000,000 lines - 3col: 182,656, 4col: 34,817,344
Processed 40,000,000 lines - 3col: 189,324, 4col: 39,810,676
Processed 45,000,000 lines - 3col: 192,973, 4col: 44,807,027
Processed 50,000,000 lines - 3col: 248,428, 4col: 49,751,572
Processed 55,000,000 lines - 3col: 292,977, 4col: 54,707,023
Processed 60,000,000 lines - 3col: 293,615, 4col: 59,706,385
Processed 65,000,000 lines - 3col: 293,770, 4col: 64,706,230
Processed 70,000,000 lines - 3col: 307,379, 4col: 69,692,621
Processed 75,000,000 lines - 3col: 433,158, 4col: 74,566,842
Processed 80,000,000 lines - 3col: 1,159,1

In [ ]:
# Selecting Sources in CPIExtract only
src_list=['BindingDB','ChEMBL','Comparative Toxicogenomics Database (CTD)','DrugBank','DrugCentral']
pc_sid=pc_sid[pc_sid['Source'].isin(src_list)]

In [ ]:
pc_sid.to_csv(data_path+'pubchem-source-SIDs.csv')

#### Get PubChem CID to InChIKey mappings

In [5]:
# Download CID-InChIKey file
url = "https://ftp.ncbi.nlm.nih.gov/pubchem/Compound/Extras/CID-InChI-Key.gz"
response = requests.head(url)
total_size = int(response.headers.get('content-length', 0))

# Download with progress bar
response = requests.get(url, stream=True)
progress_bar = tqdm(total=total_size, unit='iB', unit_scale=True, desc="Downloading")

# Save data
temp_file = os.path.join(pubchem_dir, 'CID-InChI-Key.gz')
with open(temp_file, 'wb') as f:
    for chunk in response.iter_content(chunk_size=8192):
        if chunk:
            progress_bar.update(len(chunk))
            f.write(chunk)
progress_bar.close()

print(f"Saved to {temp_file}")

4284297

In [8]:
temp_file = os.path.join(pubchem_dir, 'CID-InChI-Key.gz')

data_2col = []
with gzip.open(temp_file, 'rt') as f:
    for line_num, line in enumerate(f, 1):
        parts = line.strip().split('\t')
        data_2col.append([parts[0], parts[2]])
        
        # Progress update
        if line_num % 5000000 == 0:
            print(f"Processed {line_num:,} lines")

# Create DataFrame
pc_cid = pd.DataFrame(data_2col, columns=['CID','InChIKey'])
pc_cid['CID'] = pc_cid['CID'].astype(int)

print(f"\nDataFrame with CID mappings: {len(pc_cid):,} rows")

Processed 5,000,000 lines
Processed 10,000,000 lines
Processed 15,000,000 lines
Processed 20,000,000 lines
Processed 25,000,000 lines
Processed 30,000,000 lines
Processed 35,000,000 lines
Processed 40,000,000 lines
Processed 45,000,000 lines
Processed 50,000,000 lines
Processed 55,000,000 lines
Processed 60,000,000 lines
Processed 65,000,000 lines
Processed 70,000,000 lines
Processed 75,000,000 lines
Processed 80,000,000 lines
Processed 85,000,000 lines
Processed 90,000,000 lines
Processed 95,000,000 lines
Processed 100,000,000 lines
Processed 105,000,000 lines
Processed 110,000,000 lines
Processed 115,000,000 lines
Processed 120,000,000 lines


NameError: name 'data_3col' is not defined

In [10]:
pc_cid.to_csv(data_path+'pubchem-cid-inchikey.csv')

### Load Previously Processed PubChem Mappings

In [3]:
pc_sid=pd.read_csv(data_path+'pubchem-source-SIDs.csv',low_memory=False,index_col=0)
pc_cid=pd.read_csv(data_path+'pubchem-cid-inchikey.csv',low_memory=False,index_col=0)

### BindingDB Standardization

In [45]:
bdb_path = os.path.join(data_path, 'BindingDB_Dec2025.csv')
BDB_data = pd.read_csv(bdb_path, sep=',',low_memory=False,index_col=0)

In [46]:
# Select Source data in PubChem
pc_BDB=pc_sid[pc_sid['Source']=='BindingDB']
pc_BDB['External_ID']=pc_BDB['External_ID'].astype(int)
pc_BDB=pc_BDB.rename(columns={'External_ID':'BindingDB MonomerID'})

# Merge on External ID to link CID to each entry
BDB_data=BDB_data.merge(pc_BDB[['BindingDB MonomerID','CID']],on='BindingDB MonomerID',how='left')
BDB_data['CID']=BDB_data['CID'].fillna(-1).astype(int)

# Merge on CID to link CID to respective inchikey
BDB_data=BDB_data.merge(pc_cid[['CID','InChIKey']],on='CID',how='left')

# Ensure CPIExtract uses PubChem inchikeys
BDB_data=BDB_data.rename(columns={'inchikey':'old_inchikey','InChIKey':'inchikey'})

/tmp/ipykernel_1974477/1684305849.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pc_BDB['External_ID']=pc_BDB['External_ID'].astype(int)


In [47]:
print(f"{len(BDB_data)} compound-protein interactions in Total")
BDB_data=BDB_data[BDB_data['CID']!=-1].reset_index(drop=True)
BDB_data=BDB_data.dropna(subset=['inchikey']).reset_index(drop=True)
print(f"{len(BDB_data)} compound-protein interactions with CID and inchikey")

3078912 compound-protein interactions in Total
3071649 compound-protein interactions with CID and inchikey


In [57]:
# Get First Blocks for all compounds
BDB_data['FirstBlock']=BDB_data['inchikey'].str.split('-').str[0]

In [59]:
BDB_data.to_csv(bdb_path)

In [60]:
BDB_data.keys()

Index(['Ligand SMILES', 'Ligand InChI', 'old_inchikey', 'BindingDB MonomerID',
       'BindingDB Ligand Name',
       'Target Name Assigned by Curator or DataSource',
       'Target Source Organism According to Curator or DataSource', 'Ki (nM)',
       'IC50 (nM)', 'Kd (nM)', 'EC50 (nM)', 'pH', 'Temp (C)',
       'Curation/DataSource', 'UniProt (SwissProt) Entry Name of Target Chain',
       'UniProt (SwissProt) Primary ID of Target Chain', 'CID', 'inchikey',
       'FirstBlock'],
      dtype='object')

### ChEMBL Standardization

In [69]:
chembl_path = os.path.join(data_path, 'ChEMBL_Dec2025.csv')
chembl = pd.read_csv(chembl_path, sep=',',low_memory=False,index_col=0)

In [71]:
# Select Source data in PubChem
pc_chb=pc_sid[pc_sid['Source']=='ChEMBL']
pc_chb=pc_chb.rename(columns={'External_ID':'Molecule ChEMBL ID'})

# Merge on External ID to link CID to each entry
chembl=chembl.merge(pc_chb[['Molecule ChEMBL ID','CID']],on='Molecule ChEMBL ID',how='left')
chembl['CID']=chembl['CID'].fillna(-1).astype(int)

# Merge on CID to link CID to respective inchikey
chembl=chembl.merge(pc_cid[['CID','InChIKey']],on='CID',how='left')

# Ensure CPIExtract uses PubChem inchikeys
chembl=chembl.rename(columns={'InChIKey':'inchikey'})

In [72]:
print(f"{len(chembl)} compound-protein interactions in Total")
chembl=chembl[chembl['CID']!=-1].reset_index(drop=True)
chembl=chembl.dropna(subset=['inchikey']).reset_index(drop=True)
print(f"{len(chembl)} compound-protein interactions with CID and inchikey")

10517539 compound-protein interactions in Total
10501181 compound-protein interactions with CID and inchikey


In [73]:
# Get First Blocks for all compounds
chembl['FirstBlock']=chembl['inchikey'].str.split('-').str[0]

In [75]:
chembl.to_csv(chembl_path)

In [74]:
chembl.keys()

Index(['Molecule ChEMBL ID', 'Molecule Name', 'Smiles', 'Standard Type',
       'Standard Value', 'Standard Units', 'pChEMBL Value',
       'Data Validity Comment', 'Comment', 'Target ChEMBL ID', 'Target Name',
       'Target Organism', 'Target Type', 'Source ID', 'Action Type', 'CID',
       'inchikey', 'FirstBlock'],
      dtype='object')

### CTD Standardization

In [78]:
ctd_path=os.path.join(data_path, 'CTD_Dec2025.csv')
CTD_data=pd.read_csv(ctd_path,sep=',',low_memory=False,index_col=0)

In [83]:
# Select Source data in PubChem
pc_ctd=pc_sid[pc_sid['Source']=='Comparative Toxicogenomics Database (CTD)']
pc_ctd=pc_ctd.rename(columns={'External_ID':'ChemicalID'})

# Merge on External ID to link CID to each entry
CTD_data=CTD_data.merge(pc_ctd[['ChemicalID','CID']],on='ChemicalID',how='left')
CTD_data['CID']=CTD_data['CID'].fillna(-1).astype(int)

# Merge on CID to link CID to respective inchikey
CTD_data=CTD_data.merge(pc_cid[['CID','InChIKey']],on='CID',how='left')

# Ensure CPIExtract uses PubChem inchikeys
CTD_data=CTD_data.rename(columns={'InChIKey':'inchikey'})

In [84]:
print(f"{len(CTD_data)} compound-protein interactions in Total")
CTD_data=CTD_data[CTD_data['CID']!=-1].reset_index(drop=True)
CTD_data=CTD_data.dropna(subset=['inchikey']).reset_index(drop=True)
print(f"{len(CTD_data)} compound-protein interactions with CID and inchikey")

3019050 compound-protein interactions in Total
1639311 compound-protein interactions with CID and inchikey


In [85]:
# Get First Blocks for all compounds
CTD_data['FirstBlock']=CTD_data['inchikey'].str.split('-').str[0]

In [86]:
CTD_data.to_csv(ctd_path)

In [87]:
CTD_data.keys()

Index(['ChemicalName', 'ChemicalID', 'GeneSymbol', 'GeneID', 'GeneForms',
       'Organism', 'OrganismID', 'Interaction', 'InteractionActions', 'CID',
       'inchikey', 'FirstBlock'],
      dtype='object')

### DrugBank Standardization

In [96]:
db_path=os.path.join(data_path, 'DB_Dec2025.csv')
DB_data=pd.read_csv(db_path,low_memory=False,index_col=0)

In [97]:
# Select Source data in PubChem
pc_db=pc_sid[pc_sid['Source']=='DrugBank']
pc_db=pc_db.rename(columns={'External_ID':'drugbank-id'})

# Merge on External ID to link CID to each entry
DB_data=DB_data.merge(pc_db[['drugbank-id','CID']],on='drugbank-id',how='left')
DB_data['CID']=DB_data['CID'].fillna(-1).astype(int)

# Merge on CID to link CID to respective inchikey
DB_data=DB_data.merge(pc_cid[['CID','InChIKey']],on='CID',how='left')

# Ensure CPIExtract uses PubChem inchikeys
DB_data=DB_data.rename(columns={'InChIKey':'inchikey'})

In [98]:
print(f"{len(DB_data)} compound-protein interactions in Total")
DB_data=DB_data[DB_data['CID']!=-1].reset_index(drop=True)
DB_data=DB_data.dropna(subset=['inchikey']).reset_index(drop=True)
print(f"{len(DB_data)} compound-protein interactions with CID and inchikey")

37287 compound-protein interactions in Total
34077 compound-protein interactions with CID and inchikey


In [99]:
# Get First Blocks for all compounds
DB_data['FirstBlock']=DB_data['inchikey'].str.split('-').str[0]

In [101]:
DB_data.to_csv(db_path)

In [100]:
DB_data.keys()

Index(['drugbank-id', 'name', 'protein_name', 'protein_type', 'organism',
       'HGNC', 'CID', 'inchikey', 'FirstBlock'],
      dtype='object')

### DrugCentral Standardization

In [99]:
dc_path=os.path.join(data_path,'DrugCentral_Dec2025.csv')
DC_data=pd.read_csv(dc_path,low_memory=False,index_col=0)

In [100]:
# Select Source data in PubChem
pc_dc=pc_sid[pc_sid['Source']=='DrugCentral']
pc_dc=pc_dc.rename(columns={'External_ID':'STRUCT_ID'})

# Merge on External ID to link CID to each entry
DC_data['STRUCT_ID']=DC_data['STRUCT_ID'].astype(str)
DC_data=DC_data.merge(pc_dc[['STRUCT_ID','CID']],on='STRUCT_ID',how='left')
DC_data['CID']=DC_data['CID'].fillna(-1).astype(int)

# Merge on CID to link CID to respective inchikey
DC_data=DC_data.merge(pc_cid[['CID','InChIKey']],on='CID',how='left')

# Ensure CPIExtract uses PubChem inchikeys
DC_data=DC_data.rename(columns={'InChIKey':'inchikey'})

In [101]:
print(f"{len(DC_data)} compound-protein interactions in Total")
DC_data=DC_data[DC_data['CID']!=-1].reset_index(drop=True)
DC_data=DC_data.dropna(subset=['inchikey']).reset_index(drop=True)
print(f"{len(DC_data)} compound-protein interactions with CID and inchikey")

23115 compound-protein interactions in Total
22467 compound-protein interactions with CID and inchikey


In [102]:
# Get First Blocks for all compounds
DC_data['FirstBlock']=DC_data['inchikey'].str.split('-').str[0]

In [103]:
DC_data.to_csv(dc_path)

In [104]:
DC_data.keys()

Index(['DRUG_NAME', 'STRUCT_ID', 'TARGET_NAME', 'TARGET_CLASS', 'ACCESSION',
       'GENE', 'SWISSPROT', 'ACT_VALUE', 'ACT_UNIT', 'ACT_TYPE', 'ACT_COMMENT',
       'ACT_SOURCE', 'RELATION', 'ACTION_TYPE', 'ORGANISM', 'CID', 'inchikey',
       'FirstBlock'],
      dtype='object')

### DrugTargetCommons Standardization

In [93]:
dtc_path=os.path.join(data_path,'DTC_Dec2025.csv')
DTC_data=pd.read_csv(dtc_path,low_memory=False,index_col=0)

In [94]:
# Select Source data in PubChem
pc_chb=pc_sid[pc_sid['Source']=='ChEMBL']
pc_chb=pc_chb.rename(columns={'External_ID':'compound_id'})

# Merge on External ID to link CID to each entry
DTC_data=DTC_data.merge(pc_chb[['compound_id','CID']],on='compound_id',how='left')
DTC_data['CID']=DTC_data['CID'].fillna(-1).astype(int)

# Merge on CID to link CID to respective inchikey
DTC_data=DTC_data.merge(pc_cid[['CID','InChIKey']],on='CID',how='left')

# Ensure CPIExtract uses PubChem inchikeys
DTC_data=DTC_data.rename(columns={'InChIKey':'inchikey'})

In [95]:
print(f"{len(DTC_data)} compound-protein interactions in Total")
DTC_data=DTC_data[DTC_data['CID']!=-1].reset_index(drop=True)
DTC_data=DTC_data.dropna(subset=['inchikey']).reset_index(drop=True)
print(f"{len(DTC_data)} compound-protein interactions with CID and inchikey")

5846730 compound-protein interactions in Total
5818985 compound-protein interactions with CID and inchikey


In [96]:
# Get First Blocks for all compounds
DTC_data['FirstBlock']=DTC_data['inchikey'].str.split('-').str[0]

In [97]:
DTC_data.to_csv(dtc_path)

In [98]:
DTC_data.keys()

Index(['compound_id', 'target_id', 'gene_names', 'wildtype_or_mutant',
       'mutation_info', 'standard_type', 'standard_relation', 'standard_value',
       'standard_units', 'activity_comment', 'doc_type', 'CID', 'inchikey',
       'FirstBlock'],
      dtype='object')

### STITCH Standardization

In [76]:
sttch_path=os.path.join(data_path,'STITCH.csv')
sttch_data=pd.read_csv(sttch_path,low_memory=False,index_col=0)

In [79]:
def extract_cid(chemical_id):
    match=re.search(r'CID[a-z]*(\d+)',chemical_id)
    if match:
        return int(match.group(1))
    return None

sttch_data['CID']=sttch_data['chemical'].apply(extract_cid)

In [86]:
# Merge on CID to link CID to respective inchikey
sttch_data=sttch_data.merge(pc_cid[['CID','InChIKey']],on='CID',how='left')

# Ensure CPIExtract uses PubChem inchikeys
sttch_data=sttch_data.rename(columns={'InChIKey':'inchikey'})

In [87]:
print(f"{len(sttch_data)} compound-protein interactions in Total")
sttch_data=sttch_data[sttch_data['CID']!=-1].reset_index(drop=True)
sttch_data=sttch_data.dropna(subset=['inchikey']).reset_index(drop=True)
print(f"{len(sttch_data)} compound-protein interactions with CID and inchikey")

15473939 compound-protein interactions in Total
14974536 compound-protein interactions with CID and inchikey


In [88]:
# Get First Blocks for all compounds
sttch_data['FirstBlock']=sttch_data['inchikey'].str.split('-').str[0]

In [92]:
sttch_data.to_csv(sttch_path)

In [91]:
sttch_data.keys()

Index(['chemical', 'protein', 'experimental', 'prediction', 'database',
       'textmining', 'combined_score', 'CID', 'inchikey', 'FirstBlock'],
      dtype='object')

### OpenTargetsPlatform Standardization

In [68]:
otp_path=os.path.join(data_path,'OTP_Dec2025.csv')
OTP_data=pd.read_csv(otp_path,low_memory=False,index_col=0)

In [69]:
def parse_chembl_ids(x):
    if isinstance(x, str):
        cleaned = x.strip("[]'\"").replace("'", "").replace('"', '')
        items = [item.strip() for item in cleaned.split() if item.strip()]
        return items
    return x

OTP_data['chemblIds'] = OTP_data['chemblIds'].apply(parse_chembl_ids)
OTP_data = OTP_data.explode('chemblIds').reset_index(drop=True)

OTP_data['targets'] = OTP_data['targets'].apply(parse_chembl_ids)
OTP_data = OTP_data.explode('targets').reset_index(drop=True)

In [70]:
# Select Source data in PubChem
pc_chb=pc_sid[pc_sid['Source']=='ChEMBL']
pc_chb=pc_chb.rename(columns={'External_ID':'chemblIds'})

# Merge on External ID to link CID to each entry
OTP_data=OTP_data.merge(pc_chb[['chemblIds','CID']],on='chemblIds',how='left')
OTP_data['CID']=OTP_data['CID'].fillna(-1).astype(int)

# Merge on CID to link CID to respective inchikey
OTP_data=OTP_data.merge(pc_cid[['CID','InChIKey']],on='CID',how='left')

# Ensure CPIExtract uses PubChem inchikeys
OTP_data=OTP_data.rename(columns={'InChIKey':'inchikey'})

In [71]:
print(f"{len(OTP_data)} compound-protein interactions in Total")
OTP_data=OTP_data[OTP_data['CID']!=-1].reset_index(drop=True)
OTP_data=OTP_data.dropna(subset=['inchikey']).reset_index(drop=True)
print(f"{len(OTP_data)} compound-protein interactions with CID and inchikey")

16309 compound-protein interactions in Total
12942 compound-protein interactions with CID and inchikey


In [72]:
# Get First Blocks for all compounds
OTP_data['FirstBlock']=OTP_data['inchikey'].str.split('-').str[0]

In [75]:
OTP_data.to_csv(otp_path)

In [73]:
OTP_data.keys()

Index(['actionType', 'mechanismOfAction', 'chemblIds', 'targetName',
       'targetType', 'targets', 'CID', 'inchikey', 'FirstBlock'],
      dtype='object')

In [90]:
OTP_data

,actionType,mechanismOfAction,chemblIds,targetName,targetType,targets,CID,inchikey,FirstBlock
0,ACTIVATOR,"AMP-activated protein kinase, AMPK activator",CHEMBL1551724,"AMP-activated protein kinase, AMPK",protein complex group,ENSG00000162409,17513,RTRQQBHATOEIAF-UUOKFMHZSA-N,RTRQQBHATOEIAF
1,ACTIVATOR,"AMP-activated protein kinase, AMPK activator",CHEMBL1551724,"AMP-activated protein kinase, AMPK",protein complex group,ENSG00000131791,17513,RTRQQBHATOEIAF-UUOKFMHZSA-N,RTRQQBHATOEIAF
2,ACTIVATOR,"AMP-activated protein kinase, AMPK activator",CHEMBL1551724,"AMP-activated protein kinase, AMPK",protein complex group,ENSG00000181929,17513,RTRQQBHATOEIAF-UUOKFMHZSA-N,RTRQQBHATOEIAF
3,ACTIVATOR,"AMP-activated protein kinase, AMPK activator",CHEMBL1551724,"AMP-activated protein kinase, AMPK",protein complex group,ENSG00000106617,17513,RTRQQBHATOEIAF-UUOKFMHZSA-N,RTRQQBHATOEIAF
4,ACTIVATOR,"AMP-activated protein kinase, AMPK activator",CHEMBL1551724,"AMP-activated protein kinase, AMPK",protein complex group,ENSG00000111725,17513,RTRQQBHATOEIAF-UUOKFMHZSA-N,RTRQQBHATOEIAF
...,...,...,...,...,...,...,...,...,...
12937,SUBSTRATE,Norepinephrine transporter substrate,CHEMBL1201260,Sodium-dependent noradrenaline transporter,single protein,ENSG00000103546,2368,NIVZHWNOUVJHKV-UHFFFAOYSA-N,NIVZHWNOUVJHKV
12938,SUBSTRATE,Norepinephrine transporter substrate,CHEMBL3184143,Sodium-dependent noradrenaline transporter,single protein,ENSG00000103546,68552,RTEVGQJRTFFMLL-UHFFFAOYSA-N,RTEVGQJRTFFMLL
12939,SUBSTRATE,Norepinephrine transporter substrate,CHEMBL1037,Sodium-dependent noradrenaline transporter,single protein,ENSG00000103546,38521,HPBNRIOWIXYZFK-UHFFFAOYSA-N,HPBNRIOWIXYZFK
12940,SUBSTRATE,Serotonin transporter substrate,CHEMBL1200595,Sodium-dependent serotonin transporter,single protein,ENSG00000108576,65477,WEJDYJKJPUPMLH-UHFFFAOYSA-N,WEJDYJKJPUPMLH


In [ ]:
os.remove('pubchem_synonyms.gz')